# 🌍 AgentsVille AI Trip Planner

An AI-powered travel planning system that generates and refines vacation itineraries
using Large Language Models (LLMs), structured data validation, and tool-based reasoning.

## How It Works

1. **Collect** traveler preferences (destination, dates, interests, budget)
2. **Gather** simulated weather forecasts and available activities
3. **Generate** an initial day-by-day itinerary with the `ItineraryAgent`
4. **Evaluate** the itinerary across five automated checks
5. **Revise** the itinerary using `ItineraryRevisionAgent` (ReAct loop)
6. **Summarize** the final approved trip

---

## 🔧 Setup

Install dependencies (run once, then restart the kernel):

```bash
pip install -r requirements.txt
```

In [ ]:
import json
import os
from pathlib import Path

from openai import OpenAI

from project_lib import (
    VacationInfo,
    TravelPlan,
    get_weather_forecast,
    get_available_activities,
    run_evals,
    ItineraryAgent,
    ItineraryRevisionAgent,
    generate_trip_summary,
    print_itinerary,
    print_eval_results,
)

print('✅ Imports successful')

In [ ]:
# ── Configure your OpenAI API key ────────────────────────────────────────────
# Option A: set it here directly (not recommended for shared notebooks)
# os.environ["OPENAI_API_KEY"] = "sk-..."
#
# Option B: export it in your shell before launching Jupyter
#   export OPENAI_API_KEY="sk-..."

client = OpenAI()  # reads OPENAI_API_KEY from the environment

# Model configuration – change to gpt-4o for higher quality
MAIN_MODEL = "gpt-4o"
EVAL_MODEL = "gpt-4o-mini"

print(f'✅ OpenAI client ready  |  main model: {MAIN_MODEL}  |  eval model: {EVAL_MODEL}')

---
## Step 1 – Define Traveler Preferences

The `VacationInfo` Pydantic model captures everything the planner needs to know
about the traveler: destination, travel dates, interests, budget, and any special
constraints.

In [ ]:
vacation_info = VacationInfo(
    destination="AgentsVille",
    start_date="2026-06-10",
    end_date="2026-06-12",
    interests=["culture", "food", "outdoor activities", "entertainment"],
    budget=500.0,
    constraints=["prefer indoor options when raining"],
)

print('📋 Traveler Preferences')
print(f'   Destination : {vacation_info.destination}')
print(f'   Dates       : {vacation_info.start_date} → {vacation_info.end_date}')
print(f'   Interests   : {', '.join(vacation_info.interests)}')
print(f'   Budget      : ${vacation_info.budget:.2f}')
print(f'   Constraints : {', '.join(vacation_info.constraints) if vacation_info.constraints else "none"}')

---
## Step 2 – Data Gathering

The system simulates two API calls:

* **Weather forecast** – returns the expected weather condition for each travel date.
* **Available activities** – returns the activities that are available and
  weather-compatible for each date.

In [ ]:
# Simulate data retrieval
weather_data = get_weather_forecast(vacation_info)
available_activities = get_available_activities(vacation_info, weather_data)

print('🌤️  Weather Forecast')
for date_str, weather in sorted(weather_data.items()):
    print(f'   {date_str}: {weather}')

print()
print('🎯  Available Activities per Day')
for date_str in sorted(available_activities):
    acts = available_activities[date_str]
    print(f'   {date_str} ({weather_data[date_str]}): {len(acts)} activities available')
    for act in acts[:3]:
        print(f'      • {act["name"]}  –  ${act["cost"]:.2f}')
    if len(acts) > 3:
        print(f'      … and {len(acts) - 3} more')

---
## Step 3 – Generate Initial Itinerary

The `ItineraryAgent` sends the traveler preferences, weather forecast, and
available activities to the LLM and asks it to produce a structured
`TravelPlan` (validated by Pydantic).

In [ ]:
print('🤖  Calling ItineraryAgent…')

itinerary_agent = ItineraryAgent(client=client, model=MAIN_MODEL)
initial_plan = itinerary_agent.generate(
    vacation_info=vacation_info,
    weather_data=weather_data,
    available_activities=available_activities,
)

print_itinerary(initial_plan)

---
## Step 4 – Evaluate the Itinerary

The evaluation system runs five checks:

| Check | Type | Description |
|---|---|---|
| `budget_accuracy` | Rule-based | Costs tally correctly; total is within budget |
| `city_date_correctness` | Rule-based | Correct destination; all travel dates present |
| `minimum_activities` | Rule-based | At least 2 activities per day |
| `activity_availability` | Rule-based | All activities exist in the catalog for that date |
| `weather_compatibility` | **LLM-based** | Activities suit the day's weather |


In [ ]:
print('🔍  Running evaluations on the initial itinerary…')

eval_results = run_evals(
    plan=initial_plan,
    vacation_info=vacation_info,
    weather_data=weather_data,
    available_activities=available_activities,
    client=client,
    model=EVAL_MODEL,
)

print_eval_results(eval_results)

---
## Step 5 – Revise with the ReAct Agent

The `ItineraryRevisionAgent` follows the ReAct (Reasoning + Acting) framework:

```
THOUGHT → ACTION → OBSERVATION → repeat
```

* **THOUGHT** – the agent reasons about what needs to change
* **ACTION**  – the agent calls a tool (`get_activities_by_date_tool`,
  `calculator_tool`, `run_evals_tool`, or `final_answer_tool`)
* **OBSERVATION** – the agent reads the tool output

The loop continues until the agent calls `final_answer_tool`, signalling that
all checks pass.

In [ ]:
revision_agent = ItineraryRevisionAgent(client=client, model=MAIN_MODEL)

final_plan = revision_agent.revise(
    plan=initial_plan,
    vacation_info=vacation_info,
    weather_data=weather_data,
    available_activities=available_activities,
    eval_model=EVAL_MODEL,
)

In [ ]:
print('\n📋  Revised Itinerary')
print_itinerary(final_plan)

# Run a final evaluation to confirm
print('\n🔍  Final Evaluation')
final_eval = run_evals(
    plan=final_plan,
    vacation_info=vacation_info,
    weather_data=weather_data,
    available_activities=available_activities,
    client=client,
    model=EVAL_MODEL,
)
print_eval_results(final_eval)

---
## Step 6 – Inspect the Agent's Reasoning

The `reasoning_log` records every thought, action, and observation from the
ReAct loop so you can understand how the agent reached its decisions.

In [ ]:
print(f'\n🧠  ReAct Reasoning Log ({len(revision_agent.reasoning_log)} entries)\n')
for i, entry in enumerate(revision_agent.reasoning_log, 1):
    entry_type = entry['type'].upper()
    if entry_type == 'THOUGHT':
        print(f'[{i}] 💭 THOUGHT')
        print(f'    {entry["content"][:400].replace(chr(10), " ")}')
    elif entry_type == 'ACTION':
        print(f'[{i}] 🔧 ACTION  →  {entry["tool"]}')
        args_preview = json.dumps(entry.get('args', {}))[:200]
        print(f'    args: {args_preview}')
    elif entry_type == 'OBSERVATION':
        obs_preview = json.dumps(entry['content'])[:200]
        print(f'[{i}] 👁️  OBSERVATION')
        print(f'    {obs_preview}')
    elif entry_type == 'FINAL_ANSWER':
        print(f'[{i}] ✅ FINAL_ANSWER  –  plan submitted')
    print()

---
## Step 7 – Generate Trip Summary

Once the itinerary passes all checks, the LLM generates a short narrative
summary describing the highlights of the trip.

In [ ]:
print('✍️  Generating trip summary…\n')

summary = generate_trip_summary(
    plan=final_plan,
    vacation_info=vacation_info,
    client=client,
    model=MAIN_MODEL,
)

final_plan.summary = summary
print(summary)

---
## Step 8 – Save the Final Itinerary

Persist the approved plan as a JSON file in the `outputs/` directory.

In [ ]:
output_dir = Path('outputs')
output_dir.mkdir(exist_ok=True)

output_path = output_dir / f'itinerary_{vacation_info.destination.lower().replace(" ", "_")}_{vacation_info.start_date}.json'

with open(output_path, 'w', encoding='utf-8') as f:
    f.write(final_plan.model_dump_json(indent=2))

print(f'💾  Itinerary saved to {output_path}')
print()
print('📄  Output preview:')
print(final_plan.model_dump_json(indent=2))

---
## 🎉 Summary

This notebook demonstrated:

| Feature | Implementation |
|---|---|
| Structured input | `VacationInfo` Pydantic model |
| Simulated data | `get_weather_forecast` / `get_available_activities` |
| AI itinerary generation | `ItineraryAgent` + JSON-mode LLM output |
| Automated evaluation | 5-check eval system (rule-based + LLM) |
| Tool-based reasoning | OpenAI function-calling with 4 tools |
| ReAct loop | `ItineraryRevisionAgent` (THOUGHT→ACTION→OBSERVATION) |
| Structured output | `TravelPlan` Pydantic model |
| Narrative summary | `generate_trip_summary` |

### 🚀 Try It Yourself

Customise the `VacationInfo` in **Step 1** and re-run the notebook to plan
a completely different trip!